# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Missesqueensy/FlyRank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""## Build the feature vector

The feature vector contains five historical variables describing the recent search performance of each content page.

Features used:

- imp_last30
- imp_prev30
- clk_last30
- pos_last30
- ctr_last30

These features summarize historical Google Search Console performance and are available before the prediction date."""

'## Build the feature vector\n\nThe feature vector contains five historical variables describing the recent search performance of each content page.\n\nFeatures used:\n\n- imp_last30\n- imp_prev30\n- clk_last30\n- pos_last30\n- ctr_last30\n\nThese features summarize historical Google Search Console performance and are available before the prediction date.'

In [2]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

NameError: name 'con' is not defined

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""## Feature notes

### imp_last30
Total Google Search impressions during the last 30 days.
Missing values: replaced with 0 if necessary.
Available before prediction: Yes.

### imp_prev30
Total impressions during the previous 30-day window.
Missing values: replaced with 0.
Available before prediction: Yes.

### clk_last30
Total clicks during the last 30 days.
Missing values: replaced with 0.
Available before prediction: Yes.

### pos_last30
Average search position during the last 30 days.
Missing values: ignored by the SQL AVG function.
Available before prediction: Yes.

### ctr_last30
Average click-through rate during the last 30 days.
Missing values: replaced with 0.
Available before prediction: Yes."""

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""## The leakage hunt

I intentionally created one leaked feature derived from the target label.

As expected, model performance increased unrealistically because the leaked feature contained future information.

The leaked feature was removed immediately after the experiment."""

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""## What I excluded and why

Future clicks
Excluded because they are not available when making a prediction.

Future impressions
Excluded because they contain future information.

Trend labels
Excluded because they are derived from the prediction target.

Future CTR
Excluded because it leaks future performance.

Any feature calculated from the prediction window
Excluded to prevent data leakage and ensure honest model evaluation."""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.